In [6]:
from io import StringIO
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, davies_bouldin_score


In [7]:
USGS_ALL_MONTH = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_month.csv"
EARTH_RADIUS_KM = 6371.0088


In [8]:
def download_raw_usgs(url=USGS_ALL_MONTH) -> pd.DataFrame:
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    return pd.read_csv(StringIO(r.text))

df_raw = download_raw_usgs()
print("RAW shape:", df_raw.shape)
df_raw.to_csv("raw_all_month.csv", index=False)
print("Saved raw_all_month.csv")
df_raw.head()

RAW shape: (10011, 22)
Saved raw_all_month.csv


,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,...,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource
0,2026-02-20T16:17:53.742Z,31.6480,-104.359000,3.4028,1.900000,ml,28,71,0.1000,0.40,...,2026-02-20T16:20:53.607Z,"58 km S of Whites City, New Mexico",earthquake,0.00,1.111729,0.100000,19.0,automatic,tx,tx
1,2026-02-20T16:07:23.012Z,63.1750,-150.633000,122.3000,1.700000,ml,16,59,0.7000,0.30,...,2026-02-20T16:09:04.559Z,"68 km SE of Denali National Park, Alaska",earthquake,6.90,5.007700,0.100000,5.0,automatic,ak,ak
2,2026-02-20T15:45:13.167Z,31.6280,-104.452000,3.9664,1.500000,ml,17,102,0.0000,0.40,...,2026-02-20T16:00:25.447Z,"61 km S of Whites City, New Mexico",earthquake,0.00,1.514456,0.100000,13.0,automatic,tx,tx
3,2026-02-20T15:35:37.560Z,17.9800,-66.328833,12.6600,2.430000,md,18,151,0.1019,0.21,...,2026-02-20T16:26:21.780Z,"1 km WSW of Las Ochenta, Puerto Rico",earthquake,0.55,0.460000,0.236382,14.0,reviewed,pr,pr
4,2026-02-20T15:22:33.820Z,35.5495,-119.525665,26.4800,2.377181,ml,32,126,0.2544,0.32,...,2026-02-20T15:26:01.863Z,"17 km NNW of Buttonwillow, CA",earthquake,0.61,0.890000,0.449593,30.0,automatic,ci,ci


In [9]:
# --- Initial Data Inspection (USGS Earthquakes) ---

print("\n--- Initial Data (Head) ---")
display(df_raw.head())

# Columns used in this project (safe even if some columns are missing)
selected_cols = ["id", "time", "latitude", "longitude", "depth", "mag", "place", "type"]
selected_cols = [c for c in selected_cols if c in df_raw.columns]
df_selected = df_raw[selected_cols].copy()

print("\n--- Initial Data (Info for Selected Features) ---")
print(df_selected.info())

print("\n--- Initial Data (Missing Values for Selected Features) ---")
print(df_selected.isnull().sum())

print("\n--- Initial Data (Basic Shape) ---")
print("Rows, Columns:", df_raw.shape)
print("Selected Columns:", df_selected.shape)

# Extra inspection (recommended for report quality)
if "type" in df_selected.columns:
    print("\n--- Event Type Counts (Top 10) ---")
    print(df_selected["type"].value_counts().head(10))

if all(c in df_selected.columns for c in ["mag", "depth"]):
    print("\n--- Summary Stats (mag & depth) ---")
    print(df_selected[["mag", "depth"]].describe())


--- Initial Data (Head) ---


,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,...,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource
0,2026-02-20T16:17:53.742Z,31.6480,-104.359000,3.4028,1.900000,ml,28,71,0.1000,0.40,...,2026-02-20T16:20:53.607Z,"58 km S of Whites City, New Mexico",earthquake,0.00,1.111729,0.100000,19.0,automatic,tx,tx
1,2026-02-20T16:07:23.012Z,63.1750,-150.633000,122.3000,1.700000,ml,16,59,0.7000,0.30,...,2026-02-20T16:09:04.559Z,"68 km SE of Denali National Park, Alaska",earthquake,6.90,5.007700,0.100000,5.0,automatic,ak,ak
2,2026-02-20T15:45:13.167Z,31.6280,-104.452000,3.9664,1.500000,ml,17,102,0.0000,0.40,...,2026-02-20T16:00:25.447Z,"61 km S of Whites City, New Mexico",earthquake,0.00,1.514456,0.100000,13.0,automatic,tx,tx
3,2026-02-20T15:35:37.560Z,17.9800,-66.328833,12.6600,2.430000,md,18,151,0.1019,0.21,...,2026-02-20T16:26:21.780Z,"1 km WSW of Las Ochenta, Puerto Rico",earthquake,0.55,0.460000,0.236382,14.0,reviewed,pr,pr
4,2026-02-20T15:22:33.820Z,35.5495,-119.525665,26.4800,2.377181,ml,32,126,0.2544,0.32,...,2026-02-20T15:26:01.863Z,"17 km NNW of Buttonwillow, CA",earthquake,0.61,0.890000,0.449593,30.0,automatic,ci,ci



--- Initial Data (Info for Selected Features) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10011 entries, 0 to 10010
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         10011 non-null  object 
 1   time       10011 non-null  object 
 2   latitude   10011 non-null  float64
 3   longitude  10011 non-null  float64
 4   depth      10011 non-null  float64
 5   mag        10010 non-null  float64
 6   place      10011 non-null  object 
 7   type       10011 non-null  object 
dtypes: float64(4), object(4)
memory usage: 625.8+ KB
None

--- Initial Data (Missing Values for Selected Features) ---
id           0
time         0
latitude     0
longitude    0
depth        0
mag          1
place        0
type         0
dtype: int64

--- Initial Data (Basic Shape) ---
Rows, Columns: (10011, 22)
Selected Columns: (10011, 8)

--- Event Type Counts (Top 10) ---
type
earthquake          9864
explosion             72
quarr

In [10]:
# --- Column Selection (keep only relevant fields) ---

keep_cols = ["id", "time", "latitude", "longitude", "depth", "mag", "place", "type"]
keep_cols = [c for c in keep_cols if c in df_raw.columns] 

df_cols = df_raw[keep_cols].copy()

print("\n--- Columns Retained ---")
print(keep_cols)

print("\n--- Shape After Column Selection ---")
print("Rows, Columns:", df_cols.shape)

print("\n--- Preview After Column Selection (Head) ---")
display(df_cols.head())


--- Columns Retained ---
['id', 'time', 'latitude', 'longitude', 'depth', 'mag', 'place', 'type']

--- Shape After Column Selection ---
Rows, Columns: (10011, 8)

--- Preview After Column Selection (Head) ---


,id,time,latitude,longitude,depth,mag,place,type
0,tx2026dpasps,2026-02-20T16:17:53.742Z,31.6480,-104.359000,3.4028,1.900000,"58 km S of Whites City, New Mexico",earthquake
1,aka2026dpajqf,2026-02-20T16:07:23.012Z,63.1750,-150.633000,122.3000,1.700000,"68 km SE of Denali National Park, Alaska",earthquake
2,tx2026dozqly,2026-02-20T15:45:13.167Z,31.6280,-104.452000,3.9664,1.500000,"61 km S of Whites City, New Mexico",earthquake
3,pr71508328,2026-02-20T15:35:37.560Z,17.9800,-66.328833,12.6600,2.430000,"1 km WSW of Las Ochenta, Puerto Rico",earthquake
4,ci41400432,2026-02-20T15:22:33.820Z,35.5495,-119.525665,26.4800,2.377181,"17 km NNW of Buttonwillow, CA",earthquake


In [11]:
# --- Handling Missing Coordinates (latitude/longitude) 

before_rows = df_cols.shape[0]

missing_lat = df_cols["latitude"].isna().sum()
missing_lon = df_cols["longitude"].isna().sum()

print("--- Missing Values Before ---")
print("Missing latitude:", missing_lat)
print("Missing longitude:", missing_lon)
print("Rows before:", before_rows)

df_no_missing = df_cols.dropna(subset=["latitude", "longitude"]).copy()

after_rows = df_no_missing.shape[0]
print("\n--- After Dropping Missing Coordinates ---")
print("Rows after:", after_rows)
print("Removed rows:", before_rows - after_rows)

display(df_no_missing.head())

--- Missing Values Before ---
Missing latitude: 0
Missing longitude: 0
Rows before: 10011

--- After Dropping Missing Coordinates ---
Rows after: 10011
Removed rows: 0


,id,time,latitude,longitude,depth,mag,place,type
0,tx2026dpasps,2026-02-20T16:17:53.742Z,31.6480,-104.359000,3.4028,1.900000,"58 km S of Whites City, New Mexico",earthquake
1,aka2026dpajqf,2026-02-20T16:07:23.012Z,63.1750,-150.633000,122.3000,1.700000,"68 km SE of Denali National Park, Alaska",earthquake
2,tx2026dozqly,2026-02-20T15:45:13.167Z,31.6280,-104.452000,3.9664,1.500000,"61 km S of Whites City, New Mexico",earthquake
3,pr71508328,2026-02-20T15:35:37.560Z,17.9800,-66.328833,12.6600,2.430000,"1 km WSW of Las Ochenta, Puerto Rico",earthquake
4,ci41400432,2026-02-20T15:22:33.820Z,35.5495,-119.525665,26.4800,2.377181,"17 km NNW of Buttonwillow, CA",earthquake
